In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import joblib

# 1. Daten einlesen
meine_daten = pd.read_json("train_SBERT.json")
merkmale_X = meine_daten['text'].tolist()
zielvariable_y = meine_daten['service_name']

# 2. SBERT Modell laden
mein_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 3. Texte encodieren
x_train_zahlen = mein_sbert.encode(merkmale_X)

# 4. Cross-Validation Setup
# Wir testen, ob eine starke (0.1) oder schwache (10.0) Regularisierung besser ist
param_grid = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
basis_modell = LogisticRegression(penalty='l2', max_iter=1000, random_state=42)

# cv=5 bedeutet 5-fache Kreuzvalidierung
suche = GridSearchCV(basis_modell, param_grid, cv=5, scoring='f1_macro')

print("Starte Cross-Validation für SBERT...")
suche.fit(x_train_zahlen, zielvariable_y)

print(f"Bester Parameter für SBERT: {suche.best_params_}")
print(f"Beste Trainings-Genauigkeit: {suche.best_score_:.4f}")

# 5. Speichern des besten Modells
joblib.dump(suche.best_estimator_, 'trainiertes_Klassifikationsmodell_SBERT.joblib')
print("✅ Optimiertes SBERT-Modell gespeichert.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starte Cross-Validation für SBERT...
Bester Parameter für SBERT: {'C': 1.0}
Beste Trainings-Genauigkeit: 0.5667
✅ Optimiertes SBERT-Modell gespeichert.
